In [ ]:
import base64
import csv
import itertools
import json
import os
import re
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple

import requests

try:
    import yaml  # PyYAML
except ImportError:
    yaml = None

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None


# -----------------------------
# User paths (your exact paths)
# -----------------------------
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")
REPO_LIST_CSV = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\4 - RQ4\URL_List_Instru.csv"
)

# Output files (created next to input CSV)
OUT_SUMMARY_CSV = REPO_LIST_CSV.parent / "repo_instru_summary.csv"
OUT_RUNS_CSV = REPO_LIST_CSV.parent / "workflow_runs.csv"

# Toggle if you want to fetch runs for *all* workflows, not just instrumentation-looking ones.
FETCH_ALL_WORKFLOWS = False

# If True, also fetch per-run jobs (heavy!) and record first failed step name.
FETCH_JOB_FAILURE_DETAIL = False


# -----------------------------
# Classification heuristics
# -----------------------------
COMMUNITY_ACTION_RE = re.compile(
    r"(reactivecircus/android-emulator-runner|android-emulator-runner|usuiat/android-emulator-runner|"
    r"circus/android-emulator-runner)",
    re.IGNORECASE,
)

# Custom emulator setup (manual)
CUSTOM_EMU_RE = re.compile(
    r"(\bavdmanager\b|\bsdkmanager\b.*system-images|\bemulator\b.*\s-avd\b|\bnohup\s+emulator\b|"
    r"\badb\s+wait-for-device\b|\bemu(lator)?-headless\b)",
    re.IGNORECASE,
)

# Gradle Managed Devices signals (more reliable if 'managedDevice' appears)
GMD_RE = re.compile(r"(\bmanagedDeviceCheck\b|\ballDevicesCheck\b|\bmanagedDevice\b)", re.IGNORECASE)

# Third-party device cloud / services
THIRD_PARTY_RE = re.compile(
    r"(\bfirebase\s+test\s+android\s+run\b|\bgcloud\s+firebase\s+test\b|"
    r"\baws\b.*\bdevice\s*farm\b|\bdevicefarm\b|"
    r"\bbrowserstack\b|\bsaucelabs\b|\bkobiton\b|\bperfecto\b|\bbitbar\b|\bgenymotion\b|"
    r"\bbitrise\b|\bapp(etize|etize)\b)",
    re.IGNORECASE,
)

# Invocation hints
GRADLE_INVOKE_RE = re.compile(
    r"(^|\s)(\./gradlew|\bgradle\b)\s+([^\n\r#;]+)", re.IGNORECASE
)
ADB_INSTR_RE = re.compile(r"\badb\s+shell\s+am\s+instrument\b", re.IGNORECASE)
FIREBASE_TEST_RE = re.compile(r"\b(firebase\s+test\s+android\s+run|gcloud\s+firebase\s+test)\b", re.IGNORECASE)

# Connected instrumentation-ish gradle tasks
ANDROID_TEST_TASK_RE = re.compile(
    r"\b(connected\w*AndroidTest|connectedCheck|deviceCheck|\w*AndroidTest|managedDeviceCheck|allDevicesCheck)\b",
    re.IGNORECASE,
)


# -----------------------------
# Helpers
# -----------------------------
def load_tokens_from_env_file(env_path: Path) -> List[str]:
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")

    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)

    if not tokens:
        raise ValueError(
            f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=... through GITHUB_TOKEN_6=..."
        )
    return tokens


def parse_repo_full_name(repo_url_or_fullname: str) -> Optional[str]:
    s = repo_url_or_fullname.strip()

    # Already "owner/repo"
    if re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", s):
        return s

    # https://github.com/owner/repo or .../owner/repo.git
    m = re.search(r"github\.com[:/]+([A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+)", s, re.IGNORECASE)
    if m:
        full = m.group(1)
        return full[:-4] if full.endswith(".git") else full

    return None


def read_repo_urls(csv_path: Path) -> List[str]:
    if not csv_path.exists():
        raise FileNotFoundError(f"Repo list CSV not found: {csv_path}")

    urls: List[str] = []
    with csv_path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        reader = csv.reader(f)
        rows = list(reader)
        if not rows:
            return urls

        # Heuristic: if first row looks like header, skip it
        header = [c.strip().lower() for c in rows[0]]
        start_idx = 1 if any("url" in c or "repo" in c for c in header) else 0

        for r in rows[start_idx:]:
            if not r:
                continue
            val = (r[0] or "").strip()
            if val:
                urls.append(val)
    return urls


def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


# -----------------------------
# GitHub API Client (token rotation + pagination)
# -----------------------------
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None


class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "instru-miner/1.0",
        })
        self.tokens = [TokenState(t) for t in tokens]
        self._token_cycle = itertools.cycle(range(len(self.tokens)))
        self._idx = next(self._token_cycle)

    def _pick_token_index(self) -> int:
        now = int(time.time())
        # Prefer tokens with remaining > 0 OR unknown remaining
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None:
                candidates.append((0, i))
            elif st.remaining > 0:
                candidates.append((0, i))
            else:
                # remaining == 0, maybe reset has passed?
                if st.reset_epoch is not None and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))  # deprioritize exhausted

        candidates.sort()
        # Best candidate index
        return candidates[0][1]

    def _sleep_until_any_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch is not None]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        sleep_s = max(1, soonest - now + 2)
        print(f"[rate-limit] All tokens exhausted. Sleeping {sleep_s}s until reset...")
        time.sleep(sleep_s)

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Dict]:
        while True:
            self._idx = self._pick_token_index()
            st = self.tokens[self._idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"

            resp = self.session.request(method, url, params=params, timeout=60)

            # Update rate limit state
            try:
                st.remaining = int(resp.headers.get("X-RateLimit-Remaining", "0"))
            except ValueError:
                st.remaining = st.remaining
            try:
                st.reset_epoch = int(resp.headers.get("X-RateLimit-Reset", "0"))
            except ValueError:
                st.reset_epoch = st.reset_epoch

            if resp.status_code == 404:
                return None

            # Handle rate limiting
            if resp.status_code == 403 and "rate limit" in (resp.text or "").lower():
                # Try another token; if none, sleep
                if any((t.remaining is None) or (t.remaining > 0) or (t.reset_epoch is not None and t.reset_epoch <= int(time.time()))
                       for t in self.tokens):
                    # rotate and retry immediately
                    self._idx = next(self._token_cycle)
                    continue
                self._sleep_until_any_reset()
                continue

            # Transient server errors
            if resp.status_code in (500, 502, 503, 504):
                time.sleep(2)
                continue

            if resp.status_code >= 400:
                # Print useful debugging info and skip
                print(f"[error] {method} {url} -> {resp.status_code}")
                try:
                    print(resp.json())
                except Exception:
                    print(resp.text[:500])
                return None

            return resp.json()

    def paginate(self, url: str, params: Optional[Dict] = None, item_key: str = "") -> Iterable[Dict]:
        page = 1
        while True:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})
            data = self.request_json("GET", url, params=p)
            if not data:
                return

            items = data if isinstance(data, list) else data.get(item_key, [])
            if not items:
                return

            for it in items:
                yield it
            page += 1


# -----------------------------
# Workflow YAML download + parsing
# -----------------------------
def get_repo_metadata(gh: GitHubClient, full_name: str) -> Optional[Dict]:
    return gh.request_json("GET", f"https://api.github.com/repos/{full_name}")


def list_workflows(gh: GitHubClient, full_name: str) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/workflows"
    return list(gh.paginate(url, params={}, item_key="workflows"))


def fetch_file_contents(gh: GitHubClient, full_name: str, path: str) -> Optional[str]:
    # Contents API returns base64 content for small files; otherwise download_url.
    url = f"https://api.github.com/repos/{full_name}/contents/{path.lstrip('/')}"
    data = gh.request_json("GET", url)
    if not data:
        return None

    if isinstance(data, dict) and data.get("encoding") == "base64" and "content" in data:
        try:
            raw = base64.b64decode(data["content"]).decode("utf-8", errors="ignore")
            return raw
        except Exception:
            return None

    # Fallback: download_url if present
    dl = data.get("download_url") if isinstance(data, dict) else None
    if dl:
        # This is public raw content; ok without auth, but we reuse session
        resp = gh.session.get(dl, timeout=60)
        if resp.status_code == 200:
            return resp.text
    return None


def safe_yaml_load(text: str) -> Optional[Dict]:
    if yaml is None:
        return None
    try:
        return yaml.safe_load(text)
    except Exception:
        return None


def extract_run_blocks_from_yaml(yobj: Dict) -> List[str]:
    """Extract all step.run strings from the YAML object."""
    runs: List[str] = []
    if not isinstance(yobj, dict):
        return runs
    jobs = yobj.get("jobs", {})
    if not isinstance(jobs, dict):
        return runs
    for _, job in jobs.items():
        if not isinstance(job, dict):
            continue
        steps = job.get("steps", [])
        if not isinstance(steps, list):
            continue
        for st in steps:
            if not isinstance(st, dict):
                continue
            r = st.get("run")
            if isinstance(r, str) and r.strip():
                runs.append(r)
    return runs


def extract_uses_from_yaml(yobj: Dict) -> List[str]:
    uses: List[str] = []
    if not isinstance(yobj, dict):
        return uses
    jobs = yobj.get("jobs", {})
    if not isinstance(jobs, dict):
        return uses
    for _, job in jobs.items():
        if not isinstance(job, dict):
            continue
        steps = job.get("steps", [])
        if not isinstance(steps, list):
            continue
        for st in steps:
            if not isinstance(st, dict):
                continue
            u = st.get("uses")
            if isinstance(u, str) and u.strip():
                uses.append(u)
    return uses


def classify_workflow_text_and_hints(yaml_text: str) -> Tuple[Set[str], List[str], bool]:
    """
    Returns:
      styles: set of style labels
      invocation_hints: list of strings
      looks_like_instru: bool (should we fetch runs)
    """
    styles: Set[str] = set()
    inv: Set[str] = set()

    # Quick text-level checks
    if COMMUNITY_ACTION_RE.search(yaml_text):
        styles.add("Emu_Community_Action")
    if CUSTOM_EMU_RE.search(yaml_text):
        styles.add("Emu_Custom")
    if GMD_RE.search(yaml_text):
        styles.add("GMD")
    if THIRD_PARTY_RE.search(yaml_text):
        styles.add("ThirdParty")

    # Extract invocation hints: prefer YAML parse; fallback to regex
    yobj = safe_yaml_load(yaml_text)
    if yobj:
        for u in extract_uses_from_yaml(yobj):
            if COMMUNITY_ACTION_RE.search(u):
                inv.add(f"uses: {u}")

        for run_block in extract_run_blocks_from_yaml(yobj):
            for line in run_block.splitlines():
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                if ADB_INSTR_RE.search(line) or FIREBASE_TEST_RE.search(line) or "gradlew" in line.lower():
                    inv.add(line[:240])

    # Fallback: regex scan for gradle invocations
    for m in GRADLE_INVOKE_RE.finditer(yaml_text):
        cmd = (m.group(2) + " " + m.group(3)).strip()
        if ANDROID_TEST_TASK_RE.search(cmd) or "androidtest" in cmd.lower() or "connected" in cmd.lower():
            inv.add(cmd[:240])

    # Also capture explicit adb instrument or firebase test lines
    if ADB_INSTR_RE.search(yaml_text):
        inv.add("adb shell am instrument ...")
    if FIREBASE_TEST_RE.search(yaml_text):
        inv.add("firebase/gcloud firebase test android run ...")

    # Decide if this workflow looks instrumentation-related
    looks_like_instru = bool(
        styles
        or ANDROID_TEST_TASK_RE.search(yaml_text)
        or ADB_INSTR_RE.search(yaml_text)
        or FIREBASE_TEST_RE.search(yaml_text)
        or re.search(r"\binstrument(ation)?\b", yaml_text, re.IGNORECASE)
    )

    return styles, sorted(inv), looks_like_instru


# -----------------------------
# Runs + optional job failure detail
# -----------------------------
def list_workflow_runs(gh: GitHubClient, full_name: str, workflow_id: int) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/workflows/{workflow_id}/runs"
    return list(gh.paginate(url, params={}, item_key="workflow_runs"))


def get_run_jobs(gh: GitHubClient, full_name: str, run_id: int) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}/jobs"
    return list(gh.paginate(url, params={}, item_key="jobs"))


def first_failed_step_name(jobs_payload: List[Dict]) -> Optional[str]:
    for job in jobs_payload:
        steps = job.get("steps", [])
        if not isinstance(steps, list):
            continue
        for st in steps:
            if not isinstance(st, dict):
                continue
            if st.get("conclusion") == "failure":
                return st.get("name") or "UNKNOWN_STEP"
    return None


def iso_to_epoch(iso: Optional[str]) -> Optional[int]:
    if not iso:
        return None
    try:
        dt = datetime.fromisoformat(iso.replace("Z", "+00:00"))
        return int(dt.timestamp())
    except Exception:
        return None


# -----------------------------
# Main
# -----------------------------
def main() -> None:
    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH)
    gh = GitHubClient(tokens=tokens)

    repo_urls = read_repo_urls(REPO_LIST_CSV)
    if not repo_urls:
        raise RuntimeError(f"No repo URLs found in: {REPO_LIST_CSV}")

    # Prepare writers (append-safe)
    summary_exists = OUT_SUMMARY_CSV.exists()
    runs_exists = OUT_RUNS_CSV.exists()

    processed: Set[str] = set()
    if summary_exists:
        with OUT_SUMMARY_CSV.open("r", encoding="utf-8", errors="ignore", newline="") as f:
            rdr = csv.DictReader(f)
            for row in rdr:
                if row.get("full_name"):
                    processed.add(row["full_name"].strip())

    summary_fields = [
        "repo_url",
        "full_name",
        "is_github_actions",
        "instru_styles",
        "invocation_hints",
        "workflow_count",
        "instru_workflow_count",
        "workflow_paths_scanned",
        "scanned_at_utc",
    ]

    runs_fields = [
        "full_name",
        "workflow_id",
        "workflow_name",
        "workflow_path",
        "run_id",
        "run_number",
        "created_at",
        "updated_at",
        "status",
        "conclusion",
        "duration_seconds",
        "head_branch",
        "event",
        "html_url",
        "first_failed_step",  # only if FETCH_JOB_FAILURE_DETAIL
    ]

    if not summary_exists:
        with OUT_SUMMARY_CSV.open("w", encoding="utf-8", newline="") as f:
            csv.DictWriter(f, fieldnames=summary_fields).writeheader()

    if not runs_exists:
        with OUT_RUNS_CSV.open("w", encoding="utf-8", newline="") as f:
            csv.DictWriter(f, fieldnames=runs_fields).writeheader()

    iterator = repo_urls
    if tqdm is not None:
        iterator = tqdm(repo_urls, desc="Repos")

    for repo_url in iterator:
        full_name = parse_repo_full_name(repo_url)
        if not full_name:
            continue
        if full_name in processed:
            continue

        meta = get_repo_metadata(gh, full_name)
        if not meta:
            # repo missing or unavailable
            with OUT_SUMMARY_CSV.open("a", encoding="utf-8", newline="") as f:
                w = csv.DictWriter(f, fieldnames=summary_fields)
                w.writerow({
                    "repo_url": repo_url,
                    "full_name": full_name,
                    "is_github_actions": "UNKNOWN",
                    "instru_styles": "",
                    "invocation_hints": "",
                    "workflow_count": "0",
                    "instru_workflow_count": "0",
                    "workflow_paths_scanned": "",
                    "scanned_at_utc": now_utc_iso(),
                })
            processed.add(full_name)
            continue

        workflows = list_workflows(gh, full_name)
        is_actions = "YES" if workflows else "NO"

        all_styles: Set[str] = set()
        all_inv: Set[str] = set()
        workflow_paths: List[str] = []
        instru_workflows: List[Dict] = []

        for wf in workflows:
            wf_id = wf.get("id")
            wf_path = wf.get("path", "")
            wf_name = wf.get("name", "")
            if not wf_id or not wf_path:
                continue

            workflow_paths.append(wf_path)
            ytext = fetch_file_contents(gh, full_name, wf_path)
            if not ytext:
                continue

            styles, inv, looks_like_instru = classify_workflow_text_and_hints(ytext)
            all_styles |= styles
            all_inv |= set(inv)

            if FETCH_ALL_WORKFLOWS or looks_like_instru:
                instru_workflows.append({
                    "id": wf_id,
                    "name": wf_name,
                    "path": wf_path,
                    "looks_like_instru": looks_like_instru,
                })

        # Write summary row
        with OUT_SUMMARY_CSV.open("a", encoding="utf-8", newline="") as f:
            w = csv.DictWriter(f, fieldnames=summary_fields)
            w.writerow({
                "repo_url": repo_url,
                "full_name": full_name,
                "is_github_actions": is_actions,
                "instru_styles": ";".join(sorted(all_styles)),
                "invocation_hints": " | ".join(sorted(all_inv))[:5000],
                "workflow_count": str(len(workflows)),
                "instru_workflow_count": str(len(instru_workflows)),
                "workflow_paths_scanned": ";".join(workflow_paths)[:5000],
                "scanned_at_utc": now_utc_iso(),
            })

        # Fetch full possible run history for selected workflows
        with OUT_RUNS_CSV.open("a", encoding="utf-8", newline="") as f:
            w = csv.DictWriter(f, fieldnames=runs_fields)

            for wf in instru_workflows:
                wf_id = int(wf["id"])
                wf_name = wf.get("name", "")
                wf_path = wf.get("path", "")

                runs = list_workflow_runs(gh, full_name, wf_id)
                for run in runs:
                    created_at = run.get("created_at")
                    updated_at = run.get("updated_at")
                    dur = None
                    c_epoch = iso_to_epoch(created_at)
                    u_epoch = iso_to_epoch(updated_at)
                    if c_epoch is not None and u_epoch is not None and u_epoch >= c_epoch:
                        dur = u_epoch - c_epoch

                    fail_step = ""
                    if FETCH_JOB_FAILURE_DETAIL and run.get("conclusion") == "failure":
                        jobs = get_run_jobs(gh, full_name, int(run["id"]))
                        fs = first_failed_step_name(jobs)
                        fail_step = fs or ""

                    w.writerow({
                        "full_name": full_name,
                        "workflow_id": wf_id,
                        "workflow_name": wf_name,
                        "workflow_path": wf_path,
                        "run_id": run.get("id", ""),
                        "run_number": run.get("run_number", ""),
                        "created_at": created_at or "",
                        "updated_at": updated_at or "",
                        "status": run.get("status", ""),
                        "conclusion": run.get("conclusion", ""),
                        "duration_seconds": dur if dur is not None else "",
                        "head_branch": run.get("head_branch", ""),
                        "event": run.get("event", ""),
                        "html_url": run.get("html_url", ""),
                        "first_failed_step": fail_step,
                    })

        processed.add(full_name)

    print("Done.")
    print(f"Summary: {OUT_SUMMARY_CSV}")
    print(f"Runs:    {OUT_RUNS_CSV}")


if __name__ == "__main__":
    main()


Repos:   0%|          | 0/481 [00:00<?, ?it/s]